## get the eventhubs secret

In [ ]:
scope = "kv-northmart-gmkng"

connection_string = dbutils.secrets.get(
    scope=scope,
    key="eventhub-fraud-producer-connection-string"
)



## Configure Kafka


In [ ]:
bootstrap_servers = (
    "evhns-northmart-fraud-dev"
    ".servicebus.windows.net:9093"
)

topic = "fraud-transactions"

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": topic,

    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",

    "kafka.sasl.jaas.config": (
        'org.apache.kafka.common.security.plain.PlainLoginModule '
        'required '
        'username="$ConnectionString" '
        f'password="{connection_string}";'
    ),

    "kafka.group.id": "databricks-fraud",

    # Pour notre premier test, on veut aussi récupérer
    # les événements déjà présents dans Event Hubs.
    "startingOffsets": "earliest"
}

# Read the stream

In [ ]:
eh_namespace = "evhns-northmart-fraud-dev"
eh_name = "fraud-transactions"

bootstrap_servers = f"{eh_namespace}.servicebus.windows.net:9093"

connection_string = dbutils.secrets.get(
    scope="kv-northmart-gmkng",
    key="eventhub-fraud-producer-connection-string"
)

jaas_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    'username="$ConnectionString" '
    f'password="{connection_string}";'
)

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", eh_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    .load()
)


Pour comprendre ce que Kafka fournit, ne parse pas encore ton JSON. Sélectionne d’abord les métadonnées Kafka :

In [ ]:
bronze_stream = raw_stream.selectExpr(
    "CAST(key AS STRING) AS kafka_key",
    "CAST(value AS STRING) AS value",
    "topic",
    "partition",
    "offset",
    "timestamp AS kafka_timestamp"
)

In [ ]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS northmart_dev.bronze.checkpoints
""")

In [ ]:
# display(bronze_stream)
# you cannot display streaming flow with display !!

checkpoint_path = "/Volumes/northmart_dev/bronze/checkpoints/fraud_transactions"

query = (
    raw_stream
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .toTable("northmart_dev.bronze.fraud_transactions")
)

In [ ]:
print("isActive = ", query.isActive)
print("status = ", query.status)
print("exception = ", query.exception())

In [ ]:
df = spark.table("northmart_dev.bronze.fraud_transactions")

df.show(20, truncate=False)